In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
train=pd.read_csv('/kaggle/input/titanic/train.csv')
test=pd.read_csv('/kaggle/input/titanic/test.csv')
gen_sub=pd.read_csv('/kaggle/input/titanic/gender_submission.csv')

In [ ]:
train

In [ ]:
test

In [ ]:
p=test['PassengerId']

In [ ]:
gen_sub

In [ ]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [ ]:
Y_train=train['Survived']
Y_train

  <div style="background-color: #e7f3fe; 
            padding: 20px; 
            font: bold 30px Arial; 
            color: #31708f; 
            border: 2px solid #bce8f1; 
            border-radius: 8px;">
Visuals for pre-analysis and concept refinement.
</div>

<p style="font: 25px Arial; 
          color: black; 
          text-shadow: 2px 2px 4px #aaa;">
Survival Rate Analysis
</p>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
train["Survived"].value_counts().plot.pie(autopct="%1.1f%%", labels=["Not Survived", "Survived"], colors=["pink", "skyblue"],shadow=True)
plt.title("Overall Survival Percentage")
plt.ylabel("")  # Remove y-label for aesthetics
plt.show()


**Inference:<br> Approximately 38% passengers survived, while 62% did not, showing that more than half of the passengers perished in the Titanic disaster.**


In [ ]:
# Bar Plot: Survival Rate by Class
plt.figure(figsize=(8, 5))
ax=sns.barplot(x=train["Pclass"], y=train["Survived"]*100,hue=train['Sex'],ci=None,palette="dark:#5A9_r")
# Add value labels on top of bars
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", fontsize=9)  # Format as percentage
plt.xlabel("Passenger Class",fontsize=12)
plt.ylabel("Survival Rate %",fontsize=12)
plt.title("Survival Rate %age by Passenger Class",fontsize=14)
plt.xticks(fontsize=12)
plt.show()

**Inference: <br>Passengers in 1st class especially female, had a significantly higher survival rate compared to those in 2nd and 3rd class, indicating that passenger class influenced survival chances.**

In [ ]:

# Violin Plot: Age Distribution among Survived & Not Survived
sns.violinplot(x="Survived", y="Age", hue="Sex",data=train, palette="coolwarm", inner="quartile")
plt.title("Age Distribution of Survived & Non-Survived Passengers")
plt.xlabel('Survival')
plt.show()

**Inference:<br>Survivors were often younger and adults,especially among children, reflecting the "women and children first" policy during evacuation.**

<p style="font: 25px Arial; 
          color: black; 
          text-shadow: 2px 2px 4px #aaa;">
Class & Fare Influence
</p>

In [ ]:
# Box Plot: Fare Distribution Across Passenger Classes
sns.boxplot(x="Pclass", y="Fare", data=train, palette="pastel")
plt.title("Fare Distribution Across Classes")
plt.show()


**Inference:<br>Higher-class passengers paid more fare. The fare distribution increases significantly from 3rd to 1st class, indicating a clear economic divide on board.**


In [ ]:
# Swarm Plot: Fare vs Survival
sns.swarmplot(x="Survived", y="Fare",hue="Pclass",data=train, palette="coolwarm")
plt.title("Fare vs Survival")
plt.xlabel('Survival')
plt.show()

**Inference:<br>Passengers who paid higher fares tended to survive more, again reflecting that first-class (high fare) passengers were prioritized.**

In [ ]:
# Stacked Bar Chart: Survival by Embarkation
embarked_survival = train.groupby("Embarked")["Survived"].value_counts(normalize=True).unstack()*100
embarked_survival.plot(kind="bar", stacked=True, color=["grey", "yellow"])
plt.xlabel("Embarkation Port")
plt.ylabel("Proportion")
plt.title("Survival Rate by Embarkation Port")
plt.legend(["Not Survived", "Survived"])
plt.show()


**Inference:<br>Passengers embarking from Cherbourg (C) had the highest survival rate, and then Queenstown (Q) had the less. This may relate to class distribution by port.**


<p style="font: 25px Arial; 
          color: black; 
          text-shadow: 2px 2px 4px #aaa;">
Correlation of features
</p>

In [ ]:

# Heatmap: Correlation of Survival with Other Features
plt.figure(figsize=(8,6))
sns.heatmap(train[["Survived", "Pclass", "Fare","Age"]].corr(), annot=True, cmap="flare", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()


**Inference:<br>Fare and Pclass have a strong negative correlation.<br>Fare has a positive correlation with survival, while Pclass is negatively correlated, reinforcing earlier observations.<br>**


 <div style="background-color: #e7f3fe; 
            padding: 20px; 
            font: bold 30px Arial; 
            color: #31708f; 
            border: 2px solid #bce8f1; 
            border-radius: 8px;">
Data Preprocessing
</div>

In [ ]:
train.drop(columns='Survived',axis=1,inplace=True)
train

In [ ]:
combined = pd.concat([train, test], axis=0)

In [ ]:
combined


In [ ]:
# Reset index after merging
combined.reset_index(drop=True, inplace=True)
combined

In [ ]:

combined.sample(10)

In [ ]:
combined.shape

In [ ]:
combined.isnull().sum()

In [ ]:
combined.describe()

In [ ]:
combined.info()

In [ ]:
combined

In [ ]:
combined.nunique()

In [ ]:
combined.duplicated().sum()

In [ ]:
combined.sample(2)

In [ ]:
import matplotlib as plt 
import seaborn as sns


 <div style="background-color: #e7f3fe; 
            padding: 20px; 
            font: bold 30px Arial; 
            color: #31708f; 
            border: 2px solid #bce8f1; 
            border-radius: 8px;">
Feature Engineering
</div>

In [ ]:
combined=combined.copy()
combined.loc[:,"family_count"]=combined.iloc[:, 5]+combined.iloc[:, 6] +1

In [ ]:
combined.drop(["PassengerId","Name","Cabin","SibSp","Parch","Ticket"],axis=1,inplace=True)

In [ ]:
combined

 <div style="background-color: #e7f3fe; 
            padding: 20px; 
            font: bold 30px Arial; 
            color: #31708f; 
            border: 2px solid #bce8f1; 
            border-radius: 8px;">
Handling null values
</div>

In [ ]:
combined.isnull().sum()

In [ ]:
combined["Age"].skew()

In [ ]:
imp=combined['Age'].median()

In [ ]:
combined['Age']=combined['Age'].fillna(imp)

In [ ]:
combined['Embarked'].value_counts()

In [ ]:
combined['Embarked']=combined['Embarked'].fillna('S')

In [ ]:
combined['Fare'].skew()

In [ ]:
combined['Fare']=combined['Fare'].fillna(combined['Fare'].median())

In [ ]:
combined.isnull().sum()

In [ ]:
combined.sample(4)

## Handling data types of columns



In [ ]:
combined.info()

In [ ]:
combined['Age'].value_counts().sort_index()

In [ ]:
combined['Age']=round(combined['Age']).astype('int')

In [ ]:
combined['Age'].value_counts().sort_index()

In [ ]:
combined['Fare']=combined['Fare'].round(4)

In [ ]:
combined

In [ ]:
combined.nunique()

 <div style="background-color: #e7f3fe; 
            padding: 20px; 
            font: bold 30px Arial; 
            color: #31708f; 
            border: 2px solid #bce8f1; 
            border-radius: 8px;">
Making age groups
</div>

In [ ]:
combined['Age'].value_counts().sort_index()

In [ ]:
def categorize_age(age):
    if age <= 4:
        return "Baby"
    elif age <= 12 and age>=5:
        return "Child"
    elif age>=13 and age <= 19:
        return "Teen"
    elif age>=20 and age <= 39:
        return "Adult"
    elif age >=40 and age <= 59:
        return "Middle Age Adult"
    else:
        return "Senior Adult"

combined["Age_group"] = combined["Age"].apply(categorize_age)


In [ ]:
combined["Age_group"].value_counts()

In [ ]:
combined.pop("Age")

In [ ]:
combined

  <div style="background-color: #e7f3fe; 
            padding: 20px; 
            font: bold 30px Arial; 
            color: #31708f; 
            border: 2px solid #bce8f1; 
            border-radius: 8px;">
Time to encode categorical columns
</div>




 
<div style="background-color: #f0f0f0; 
            padding: 20px; 
            font: italic 18px Georgia; 
            color: #333; 
            border-left: 10px solid gray;">
Pclass-ordinal encoding <br>
Sex- Binary encoding<br>
Embarked- one hot encoding<br>
Age-group -ordinal
</div>


In [ ]:
from sklearn.preprocessing import OneHotEncoder,LabelEncoder, OrdinalEncoder

In [ ]:
Ohe=OneHotEncoder(drop='first',sparse_output=False)
OE=OrdinalEncoder(categories=[[1,2,3],["Baby", "Child", "Teen", "Adult", "Middle Age Adult", "Senior Adult"]])

In [ ]:
from sklearn.compose import ColumnTransformer
preprocessor=ColumnTransformer([("onehot_encoding",Ohe,["Embarked"]),
                               ("ordinal_encoding",OE,["Pclass","Age_group"])],remainder="passthrough")

In [ ]:
combined_encoded=preprocessor.fit_transform(combined)

In [ ]:
combined_encoded=pd.DataFrame(combined_encoded,columns=preprocessor.get_feature_names_out())

In [ ]:
combined_encoded['Sex']=combined['Sex'].map({"male":0,"female":1})

In [ ]:
combined_encoded

In [ ]:
combined_encoded.drop("remainder__Sex",axis=1,inplace=True)

In [ ]:
combined_encoded.rename(columns={"remainder__Fare":"Fare",	"remainder__family_count":"family_count"},inplace=True)

In [ ]:
combined_encoded


  <div style="background-color: #e7f3fe; 
            padding: 20px; 
            font: bold 30px Arial; 
            color: #31708f; 
            border: 2px solid #bce8f1; 
            border-radius: 8px;">
Feature Scaling
</div>

In [ ]:
from sklearn.preprocessing import StandardScaler
scale=StandardScaler()
scaler=ColumnTransformer([("scaling",scale,['Fare','family_count'])],remainder="passthrough")
combined_scaled=scaler.fit_transform(combined_encoded)

In [ ]:
combined_scaled=pd.DataFrame(combined_scaled,columns=scaler.get_feature_names_out())

In [ ]:
combined_scaled

In [ ]:
combined_scaled.rename(columns={"scaling__Fare":"Fare","scaling__family_count":"family_count","remainder__Sex":"Sex"},inplace=True)

In [ ]:
combined_scaled

  <div style="background-color: #e7f3fe; 
            padding: 20px; 
            font: bold 30px Arial; 
            color: #31708f; 
            border: 2px solid #bce8f1; 
            border-radius: 8px;">
 Let's split data into train and test datasets 
</div>

In [ ]:
train=combined_scaled.iloc[:891]
test=combined_scaled.iloc[891:]

In [ ]:
train

In [ ]:
test.reset_index(drop=True,inplace=True)

In [ ]:
test

<h3 style="background: linear-gradient(to right, #ff7e5f, #feb47b); 
           padding: 15px; 
           font: bold 26px Arial; 
           color: green; 
           border-radius: 8px;">   
DATA is all set for creating a model 
</h3>

In [ ]:
y_test=gen_sub['Survived']

In [ ]:
y_test



 
<div style="background-color: #f0f0f0; 
            padding: 20px; 
            font: italic 25px Georgia; 
            color: #333; 
            border-left: 10px solid gray;">
Identify the top algorithms with the highest accuracy, fine-tune them to achieve optimal performance, and determine the best algorithm.
</div>


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression,LogisticRegressionCV,SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier,AdaBoostClassifier,BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [ ]:

# Define models
models = {
    'Logistic Regression': LogisticRegression(),
        'Logistic Regression CV': LogisticRegressionCV(),
    'SGD': SGDClassifier(),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'AdaBoost': AdaBoostClassifier(),
    'Bagging': BaggingClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'Support Vector Machine': SVC(),
    'K-Nearest Neighbors': KNeighborsClassifier()
}

# Train and evaluate models
def evaluate_models(X_train, X_test, y_train, y_test):
    results = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results.append((name, acc))
    
    # Sort models by accuracy
    results.sort(key=lambda x: x[1], reverse=True)
    return results


results = evaluate_models(train,test,Y_train,y_test)
    
print("Model Performance:")
for name, acc in results:
    print(f"{name}: {acc:.6f}")


<h3 style="background: linear-gradient(to right, #ffd89b ,#19547b); 
           padding: 15px; 
           font: bold 26px Arial; 
           color: black; 
           border-radius: 8px;">   
Let's see the effect on accuracy with Support Vector Machine Model
</h3>

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import KFold, cross_val_score

In [ ]:
for c in [0.001, 0.01, 0.05,0.1,0.5, 1, 10]:
    model = SVC(kernel='linear', C=c, gamma=0.01, class_weight='balanced')
    model.fit(train, Y_train)
    Y_pred1=model.predict(test)
    print(f"C={c}, Train acc: {model.score(train, Y_train):.3f}, Test acc: {model.score(test, y_test):.3f}")


In [ ]:
svm=SVC(kernel="linear",C=0.05)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)  # 5-fold cross-validation 


In [ ]:
scores = cross_val_score(svm, train, Y_train, cv=kf, scoring='accuracy')
print("Accuracy for each fold:", scores)
print("Average Accuracy:", scores.mean())

In [ ]:
svm.fit(train,Y_train)

In [ ]:
Y_pred1=svm.predict(test)

In [ ]:
accuracy1 = accuracy_score(y_test, Y_pred1)
print("Accuracy Score is :", accuracy1)

In [ ]:
print(f"C=0.05, Train acc: {svm.score(train, Y_train):.3f}, Test acc: {svm.score(test, y_test):.3f}")


#### Model is not working upto the mark

<h3 style="background: linear-gradient(to right, #ffd89b ,#19547b); 
           padding: 15px; 
           font: bold 26px Arial; 
           color: black; 
           border-radius: 8px;">   
Let's see the effect on accuracy with Logistic Regression Model
</h3>

In [ ]:
lr=LogisticRegression(penalty='l1',C=0.1,max_iter=160,solver='liblinear')

In [ ]:
lr.fit(train,Y_train)

In [ ]:
Y_pred2=lr.predict(test)

In [ ]:
Y_pred2

In [ ]:
accuracy2 = accuracy_score(y_test, Y_pred2)
print("Accuracy Score is :", accuracy2)

<h3 style="background: linear-gradient(to right, #ffd89b ,#19547b); 
           padding: 15px; 
           font: bold 26px Arial; 
           color: black; 
           border-radius: 8px;">   
Let's see the effect on accuracy with AdaBoost Model
</h3>

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# Define the base estimator
base_estimator = DecisionTreeClassifier(random_state=42)

# Create AdaBoost model
ada = AdaBoostClassifier(estimator=base_estimator, random_state=42)

# Define parameter grid
param_grid = {
    'n_estimators': [50, 80,100,120, 150,180],
    'learning_rate': [0.01,0.05, 0.1, 0.5, 1],
    'estimator__max_depth': [1, 2, 3,4]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=ada, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit on training data
grid_search.fit(train, Y_train)

# Best parameters
print("Best Parameters:", grid_search.best_params_)

# Predict using best estimator
best_model = grid_search.best_estimator_
Y_pred3 = best_model.predict(test)

# Accuracy
from sklearn.metrics import accuracy_score
print("Accuracy:", accuracy_score(y_test, Y_pred3))


## FINAL SUBMISSION WITH BEST ACCURACY

In [ ]:
submission=pd.DataFrame([pd.Series(p),pd.Series(Y_pred3)],columns=['PassengerId','Survived'])

In [ ]:
submission = pd.DataFrame({
    'PassengerId': p,  # Extracting PassengerId from test dataset
    'Survived': Y_pred3  # Predicted values from model
})

In [ ]:
submission

In [ ]:
submission.to_csv("submission.csv", index=False)

<h3 style = "background-color: #000033;
             padding: 15px;
             font: bold 32px arial;
             color: #ccebff;
             border: 2px #e6e6ff;
             border-radius: 8px">
Enjoyed the notebook? An upvote would be truly appreciated!</h3>